<a href="https://colab.research.google.com/github/megamiro-code/battlefield/blob/main/%E5%85%B5%E5%A3%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from IPython.display import HTML, display

display(HTML("""
<iframe
    srcdoc='
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">

<style>
html, body {
    margin: 0;
    width: 100%;
    height: 100%;
    overflow: hidden;
    background: #111;
}

#info, #debug {
    position: absolute;
    z-index: 10;
    color: white;
    background: rgba(0,0,0,0.82);
    padding: 10px 14px;
    font-family: monospace;
    font-size: 13px;
    border-radius: 6px;
    line-height: 1.45;
}

#info {
    top: 10px;
    left: 10px;
}

#debug {
    top: 10px;
    right: 10px;
    min-width: 320px;
}
</style>
</head>

<body>

<div id="info">
<b>10 × 10 Battlefield</b><br>
🔴 Red Army : 100 Soldiers + Commander<br>
🔵 Blue Army : 100 Soldiers + Commander<br>
<b>Commander AI: Neural Network</b><br>
<span id="status">Battle ongoing...</span>
</div>

<div id="debug">
<b>DEBUG</b><br>

FPS:
<span id="fps">0</span><br>

Time:
<span id="time">0.0</span> s<br>

Remaining:
<span id="remaining">30.0</span> s<br><br>

Red alive:
<span id="redAlive">0</span><br>

Blue alive:
<span id="blueAlive">0</span><br><br>

Red Commander HP:
<span id="redHP">0</span><br>

Blue Commander HP:
<span id="blueHP">0</span><br><br>

Attacks:
<span id="attacks">0</span><br>

Blocked moves:
<span id="blocked">0</span><br>

Stuck units:
<span id="stuck">0</span><br><br>

NN command updates:
<span id="commandUpdates">0</span><br>

NN forward passes:
<span id="forwardPasses">0</span><br><br>

<b>REWARD</b><br>

Red reward:
<span id="redReward">0</span><br>

Blue reward:
<span id="blueReward">0</span><br>

Episode reward:
<span id="episodeReward">0</span>
</div>


<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>

<script>

// ============================================================
// SETTINGS
// ============================================================

const FIELD_SIZE = 10;
const FIELD_LIMIT = FIELD_SIZE / 2;

const SOLDIER_RADIUS = 0.13;
const COMMANDER_RADIUS = 0.22;

const COMMANDER_SAFETY_MARGIN = 0.10;


// ------------------------------------------------------------
// HP
// ------------------------------------------------------------

const SOLDIER_HP = 100;
const COMMANDER_HP = 200;


// ------------------------------------------------------------
// Combat
// ------------------------------------------------------------

const ATTACK_RANGE = 0.38;
const ATTACK_DAMAGE = 25;
const ATTACK_COOLDOWN = 0.55;


// ------------------------------------------------------------
// Movement
// ------------------------------------------------------------

const SOLDIER_SPEED_MIN = 0.40;
const SOLDIER_SPEED_MAX = 0.65;

const COMMANDER_SPEED = 0.20;


// ------------------------------------------------------------
// Neural network
// ------------------------------------------------------------

// 202 units × 10 features
const NUM_INPUTS = 202 * 10;

// Hidden layers
const HIDDEN1 = 512;
const HIDDEN2 = 256;

// 100 soldiers × 3 outputs
// dx, dz, attack
const NUM_OUTPUTS = 100 * 3;


// ------------------------------------------------------------
// Commander update
// ------------------------------------------------------------

const COMMAND_INTERVAL = 0.15;


// ------------------------------------------------------------
// Battle time limit
// ------------------------------------------------------------

const MAX_BATTLE_TIME = 30.0;


// ============================================================
// DEBUG
// ============================================================

let totalAttacks = 0;

let totalBlockedMoves = 0;

let commandUpdates = 0;

let forwardPasses = 0;


// ============================================================
// REWARD
// ============================================================

let redReward = 0;

let blueReward = 0;

let episodeReward = 0;


// ============================================================
// BATTLE
// ============================================================

let battleEnded = false;

let battleEndReason = "";

let battleWinner = null;

let battleStartTime =
    performance.now();


// ============================================================
// SCENE
// ============================================================

const scene =
    new THREE.Scene();

scene.background =
    new THREE.Color(
        0x20242a
    );


// ============================================================
// CAMERA
// ============================================================

const camera =
    new THREE.PerspectiveCamera(
        50,
        window.innerWidth /
        window.innerHeight,
        0.1,
        100
    );

camera.position.set(
    0,
    15,
    5
);

camera.lookAt(
    0,
    0,
    0
);


// ============================================================
// RENDERER
// ============================================================

const renderer =
    new THREE.WebGLRenderer({
        antialias: true
    });

renderer.setPixelRatio(
    window.devicePixelRatio
);

renderer.setSize(
    window.innerWidth,
    window.innerHeight
);

document.body.appendChild(
    renderer.domElement
);


// ============================================================
// LIGHT
// ============================================================

scene.add(
    new THREE.AmbientLight(
        0xffffff,
        0.8
    )
);

const light =
    new THREE.DirectionalLight(
        0xffffff,
        1
    );

light.position.set(
    5,
    12,
    5
);

scene.add(light);


// ============================================================
// GROUND
// ============================================================

const ground =
    new THREE.Mesh(
        new THREE.BoxGeometry(
            FIELD_SIZE,
            0.2,
            FIELD_SIZE
        ),
        new THREE.MeshStandardMaterial({
            color: 0xd9d9d9
        })
    );

ground.position.y =
    -0.1;

scene.add(ground);


// ============================================================
// GRID
// ============================================================

for (
    let i = 0;
    i <= FIELD_SIZE;
    i++
) {

    const p =
        -FIELD_LIMIT + i;

    const material =
        new THREE.LineBasicMaterial({
            color: 0x888888
        });


    // X direction
    const geoX =
        new THREE.BufferGeometry();

    geoX.setFromPoints([
        new THREE.Vector3(
            p,
            0.01,
            -FIELD_LIMIT
        ),
        new THREE.Vector3(
            p,
            0.01,
            FIELD_LIMIT
        )
    ]);

    scene.add(
        new THREE.Line(
            geoX,
            material
        )
    );


    // Z direction
    const geoZ =
        new THREE.BufferGeometry();

    geoZ.setFromPoints([
        new THREE.Vector3(
            -FIELD_LIMIT,
            0.01,
            p
        ),
        new THREE.Vector3(
            FIELD_LIMIT,
            0.01,
            p
        )
    ]);

    scene.add(
        new THREE.Line(
            geoZ,
            material.clone()
        )
    );
}


// ============================================================
// WALLS
// ============================================================

const WALL_SIZE = 1.0;

const WALL_HEIGHT = 0.7;

const wallPositions = [

    // Four corners
    [-4.5, -4.5],
    [-4.5,  4.5],
    [ 4.5, -4.5],
    [ 4.5,  4.5],

    // Center 2 × 2
    [-0.5, -0.5],
    [-0.5,  0.5],
    [ 0.5, -0.5],
    [ 0.5,  0.5]
];

const walls = [];


for (
    const [x, z]
    of wallPositions
) {

    const wall =
        new THREE.Mesh(
            new THREE.BoxGeometry(
                WALL_SIZE,
                WALL_HEIGHT,
                WALL_SIZE
            ),
            new THREE.MeshStandardMaterial({
                color: 0x777777
            })
        );


    wall.position.set(
        x,
        WALL_HEIGHT / 2,
        z
    );


    scene.add(wall);


    walls.push({
        x: x,
        z: z,
        halfSize:
            WALL_SIZE / 2
    });
}


// ============================================================
// SOLDIER MODEL
// ============================================================

function createSoldier(color) {

    const group =
        new THREE.Group();


    const body =
        new THREE.Mesh(
            new THREE.BoxGeometry(
                0.22,
                0.28,
                0.22
            ),
            new THREE.MeshStandardMaterial({
                color: color
            })
        );

    body.position.y =
        0.15;

    group.add(body);


    const head =
        new THREE.Mesh(
            new THREE.BoxGeometry(
                0.16,
                0.16,
                0.16
            ),
            new THREE.MeshStandardMaterial({
                color: color
            })
        );

    head.position.y =
        0.37;

    group.add(head);


    return group;
}


// ============================================================
// COMMANDER MODEL
// ============================================================

function createCommander(color) {

    const group =
        new THREE.Group();


    const body =
        new THREE.Mesh(
            new THREE.BoxGeometry(
                0.4,
                0.5,
                0.4
            ),
            new THREE.MeshStandardMaterial({
                color: color
            })
        );

    body.position.y =
        0.25;

    group.add(body);


    const head =
        new THREE.Mesh(
            new THREE.BoxGeometry(
                0.3,
                0.3,
                0.3
            ),
            new THREE.MeshStandardMaterial({
                color: color
            })
        );

    head.position.y =
        0.65;

    group.add(head);


    const crown =
        new THREE.Mesh(
            new THREE.ConeGeometry(
                0.20,
                0.20,
                4
            ),
            new THREE.MeshStandardMaterial({
                color: 0xffd700
            })
        );

    crown.position.y =
        0.90;

    group.add(crown);


    return group;
}


// ============================================================
// UNITS
// ============================================================

const units = [];


// ============================================================
// UTILITY
// ============================================================

function distance2D(
    x1,
    z1,
    x2,
    z2
) {

    const dx =
        x1 - x2;

    const dz =
        z1 - z2;

    return Math.sqrt(
        dx * dx +
        dz * dz
    );
}


// ============================================================
// WALL COLLISION
// ============================================================

function collidesWithWall(
    x,
    z,
    radius
) {

    for (
        const wall
        of walls
    ) {

        const minX =
            wall.x -
            wall.halfSize -
            radius;

        const maxX =
            wall.x +
            wall.halfSize +
            radius;

        const minZ =
            wall.z -
            wall.halfSize -
            radius;

        const maxZ =
            wall.z +
            wall.halfSize +
            radius;


        if (
            x >= minX &&
            x <= maxX &&
            z >= minZ &&
            z <= maxZ
        ) {
            return true;
        }
    }

    return false;
}


// ============================================================
// UNIT COLLISION
// ============================================================

function collidesWithUnit(
    x,
    z,
    radius,
    ignoreUnit = null
) {

    for (
        const unit
        of units
    ) {

        if (
            unit === ignoreUnit ||
            !unit.userData.alive
        ) {
            continue;
        }


        let requiredDistance =
            radius +
            unit.userData.radius +
            0.02;


        if (
            unit.userData.type ===
            "commander"
        ) {

            requiredDistance +=
                COMMANDER_SAFETY_MARGIN;
        }


        const d =
            distance2D(
                x,
                z,
                unit.position.x,
                unit.position.z
            );


        if (
            d <
            requiredDistance
        ) {
            return true;
        }
    }

    return false;
}


// ============================================================
// INITIAL POSITION
// ============================================================

function findFreePosition(
    minX,
    maxX,
    minZ,
    maxZ,
    radius
) {

    for (
        let attempt = 0;
        attempt < 5000;
        attempt++
    ) {

        const x =
            minX +
            Math.random() *
            (
                maxX -
                minX
            );

        const z =
            minZ +
            Math.random() *
            (
                maxZ -
                minZ
            );


        if (
            !collidesWithWall(
                x,
                z,
                radius
            ) &&
            !collidesWithUnit(
                x,
                z,
                radius
            )
        ) {

            return {
                x: x,
                z: z
            };
        }
    }


    throw new Error(
        "Could not find a free initial position."
    );
}


// ============================================================
// COMMANDERS
// ============================================================

function createCommanderUnit(
    color,
    team,
    x,
    z
) {

    const commander =
        createCommander(color);


    commander.position.set(
        x,
        0,
        z
    );


    commander.userData = {

        team: team,

        type: "commander",

        hp: COMMANDER_HP,

        maxHp: COMMANDER_HP,

        alive: true,

        radius:
            COMMANDER_RADIUS,

        velocity: {
            x: 0,
            z: 0
        },

        speed:
            COMMANDER_SPEED,

        attackCooldown: 0
    };


    scene.add(commander);

    units.push(commander);

    return commander;
}


// 大将を先に配置
const redCommander =
    createCommanderUnit(
        0xff0000,
        "red",
        -3.8,
        0
    );


const blueCommander =
    createCommanderUnit(
        0x0066ff,
        "blue",
        3.8,
        0
    );


// ============================================================
// SOLDIERS
// ============================================================

function createMovingSoldier(
    color,
    team,
    minX,
    maxX,
    minZ,
    maxZ
) {

    const radius =
        SOLDIER_RADIUS;


    const pos =
        findFreePosition(
            minX,
            maxX,
            minZ,
            maxZ,
            radius
        );


    const soldier =
        createSoldier(color);


    soldier.position.set(
        pos.x,
        0,
        pos.z
    );


    soldier.userData = {

        team: team,

        type: "soldier",

        hp: SOLDIER_HP,

        maxHp: SOLDIER_HP,

        alive: true,

        radius: radius,

        command: {
            dx: 0,
            dz: 0,
            attack: false
        },

        speed:
            SOLDIER_SPEED_MIN +
            Math.random() *
            (
                SOLDIER_SPEED_MAX -
                SOLDIER_SPEED_MIN
            ),

        attackCooldown: 0,

        stuckTime: 0
    };


    scene.add(soldier);

    units.push(soldier);

    return soldier;
}


// Red 100
for (
    let i = 0;
    i < 100;
    i++
) {

    createMovingSoldier(
        0xff3333,
        "red",
        -4.3,
        -0.8,
        -4.3,
        4.3
    );
}


// Blue 100
for (
    let i = 0;
    i < 100;
    i++
) {

    createMovingSoldier(
        0x3388ff,
        "blue",
        0.8,
        4.3,
        -4.3,
        4.3
    );
}


// ============================================================
// NORMAL RANDOM NUMBER
// ============================================================

function randomNormal() {

    let u = 0;
    let v = 0;


    while (
        u === 0
    ) {
        u = Math.random();
    }


    while (
        v === 0
    ) {
        v = Math.random();
    }


    return Math.sqrt(
        -2.0 *
        Math.log(u)
    ) *
    Math.cos(
        2.0 *
        Math.PI *
        v
    );
}


// ============================================================
// MATRIX
// ============================================================

function createMatrix(
    rows,
    cols
) {

    const data =
        new Float32Array(
            rows * cols
        );


    const std =
        Math.sqrt(
            2 /
            (
                rows +
                cols
            )
        );


    for (
        let i = 0;
        i < data.length;
        i++
    ) {

        data[i] =
            randomNormal() *
            std;
    }


    return {
        rows: rows,
        cols: cols,
        data: data
    };
}


function createVector(
    size
) {

    return new Float32Array(
        size
    );
}


// ============================================================
// MLP
// 2020 → 512 → 256 → 300
// ============================================================

class CommanderNetwork {

    constructor() {

        this.W1 =
            createMatrix(
                HIDDEN1,
                NUM_INPUTS
            );

        this.b1 =
            createVector(
                HIDDEN1
            );


        this.W2 =
            createMatrix(
                HIDDEN2,
                HIDDEN1
            );

        this.b2 =
            createVector(
                HIDDEN2
            );


        this.W3 =
            createMatrix(
                NUM_OUTPUTS,
                HIDDEN2
            );

        this.b3 =
            createVector(
                NUM_OUTPUTS
            );
    }


    forward(input) {

        // ----------------------------------------------------
        // Layer 1
        // ----------------------------------------------------

        const h1 =
            new Float32Array(
                HIDDEN1
            );


        for (
            let i = 0;
            i < HIDDEN1;
            i++
        ) {

            let sum =
                this.b1[i];


            const rowStart =
                i *
                NUM_INPUTS;


            for (
                let j = 0;
                j < NUM_INPUTS;
                j++
            ) {

                sum +=
                    this.W1.data[
                        rowStart + j
                    ] *
                    input[j];
            }


            h1[i] =
                sum > 0
                    ? sum
                    : 0;
        }


        // ----------------------------------------------------
        // Layer 2
        // ----------------------------------------------------

        const h2 =
            new Float32Array(
                HIDDEN2
            );


        for (
            let i = 0;
            i < HIDDEN2;
            i++
        ) {

            let sum =
                this.b2[i];


            const rowStart =
                i *
                HIDDEN1;


            for (
                let j = 0;
                j < HIDDEN1;
                j++
            ) {

                sum +=
                    this.W2.data[
                        rowStart + j
                    ] *
                    h1[j];
            }


            h2[i] =
                sum > 0
                    ? sum
                    : 0;
        }


        // ----------------------------------------------------
        // Output
        // ----------------------------------------------------

        const output =
            new Float32Array(
                NUM_OUTPUTS
            );


        for (
            let i = 0;
            i < NUM_OUTPUTS;
            i++
        ) {

            let sum =
                this.b3[i];


            const rowStart =
                i *
                HIDDEN2;


            for (
                let j = 0;
                j < HIDDEN2;
                j++
            ) {

                sum +=
                    this.W3.data[
                        rowStart + j
                    ] *
                    h2[j];
            }


            output[i] =
                sum;
        }


        return output;
    }
}


// ============================================================
// TWO COMMANDER NETWORKS
// ============================================================

const redNetwork =
    new CommanderNetwork();

const blueNetwork =
    new CommanderNetwork();


// ============================================================
// AI INPUT
// ============================================================
//
// [x, z,
//  vx, vz,
//  HP,
//  team_self,
//  team_enemy,
//  soldier,
//  commander,
//  alive]
//
// ============================================================

function buildAIInput(
    perspectiveTeam
) {

    const input =
        new Float32Array(
            NUM_INPUTS
        );


    for (
        let i = 0;
        i < units.length;
        i++
    ) {

        const unit =
            units[i];


        const base =
            i * 10;


        // Position
        input[base + 0] =
            unit.position.x /
            FIELD_LIMIT;

        input[base + 1] =
            unit.position.z /
            FIELD_LIMIT;


        // Velocity
        input[base + 2] =
            unit.userData.velocity
                ? unit.userData.velocity.x
                : 0;

        input[base + 3] =
            unit.userData.velocity
                ? unit.userData.velocity.z
                : 0;


        // HP
        input[base + 4] =
            unit.userData.hp /
            unit.userData.maxHp;


        // Team
        if (
            unit.userData.team ===
            perspectiveTeam
        ) {

            input[base + 5] =
                1;

            input[base + 6] =
                0;

        } else {

            input[base + 5] =
                0;

            input[base + 6] =
                1;
        }


        // Type
        if (
            unit.userData.type ===
            "soldier"
        ) {

            input[base + 7] =
                1;

            input[base + 8] =
                0;

        } else {

            input[base + 7] =
                0;

            input[base + 8] =
                1;
        }


        // Alive
        input[base + 9] =
            unit.userData.alive
                ? 1
                : 0;
    }


    return input;
}


// ============================================================
// SIGMOID
// ============================================================

function sigmoid(x) {

    if (
        x < -20
    ) {
        return 0;
    }


    if (
        x > 20
    ) {
        return 1;
    }


    return 1 /
        (
            1 +
            Math.exp(-x)
        );
}


// ============================================================
// APPLY NN
// ============================================================

function applyNetworkCommands(
    network,
    team
) {

    const input =
        buildAIInput(
            team
        );


    const output =
        network.forward(
            input
        );


    forwardPasses++;


    const soldiers = [];


    for (
        const unit
        of units
    ) {

        if (
            unit.userData.team ===
            team &&
            unit.userData.type ===
            "soldier"
        ) {

            soldiers.push(unit);
        }
    }


    for (
        let i = 0;
        i < soldiers.length;
        i++
    ) {

        const unit =
            soldiers[i];


        const base =
            i * 3;


        // dx
        let dx =
            Math.tanh(
                output[
                    base + 0
                ]
            );


        // dz
        let dz =
            Math.tanh(
                output[
                    base + 1
                ]
            );


        // Normalize movement vector
        const length =
            Math.sqrt(
                dx * dx +
                dz * dz
            );


        if (
            length > 0.0001
        ) {

            dx /= length;
            dz /= length;
        }


        // Attack
        const attackProbability =
            sigmoid(
                output[
                    base + 2
                ]
            );


        const attack =
            attackProbability >
            0.5;


        unit.userData.command = {

            dx: dx,

            dz: dz,

            attack: attack
        };
    }
}


// ============================================================
// UPDATE COMMANDS
// ============================================================

function updateNNCommands() {

    applyNetworkCommands(
        redNetwork,
        "red"
    );


    applyNetworkCommands(
        blueNetwork,
        "blue"
    );


    commandUpdates++;
}


// ============================================================
// FIND NEAREST ENEMY
// ============================================================

function findNearestEnemy(
    unit
) {

    let nearest = null;

    let nearestDistance =
        Infinity;


    for (
        const other
        of units
    ) {

        if (
            !other.userData.alive ||
            other.userData.team ===
            unit.userData.team
        ) {
            continue;
        }


        const d =
            distance2D(
                unit.position.x,
                unit.position.z,
                other.position.x,
                other.position.z
            );


        if (
            d <
            nearestDistance
        ) {

            nearest =
                other;

            nearestDistance =
                d;
        }
    }


    return {
        unit: nearest,
        distance: nearestDistance
    };
}


// ============================================================
// ATTACK
// ============================================================

function attack(
    attacker,
    target
) {

    if (
        !attacker.userData.alive ||
        !target.userData.alive
    ) {
        return;
    }


    if (
        attacker.userData.attackCooldown >
        0
    ) {
        return;
    }


    target.userData.hp -=
        ATTACK_DAMAGE;


    attacker.userData.attackCooldown =
        ATTACK_COOLDOWN;


    totalAttacks++;


    if (
        target.userData.hp <= 0
    ) {

        target.userData.hp =
            0;

        target.userData.alive =
            false;

        target.visible =
            false;


        // Commander death
        if (
            target.userData.type ===
            "commander"
        ) {

            endBattle(
                attacker.userData.team,
                "commander"
            );
        }
    }
}


// ============================================================
// COUNT ALIVE SOLDIERS
// ============================================================

function countAliveSoldiers(
    team
) {

    let count = 0;


    for (
        const unit
        of units
    ) {

        if (
            unit.userData.alive &&
            unit.userData.type ===
            "soldier" &&
            unit.userData.team ===
            team
        ) {

            count++;
        }
    }


    return count;
}


// ============================================================
// END BATTLE
// ============================================================
//
// reason:
// "commander" = 大将撃破
// "timeout"   = 時間切れ
// ============================================================

function endBattle(
    winner,
    reason
) {

    if (
        battleEnded
    ) {
        return;
    }


    battleEnded = true;

    battleWinner =
        winner;

    battleEndReason =
        reason;


    // ========================================================
    // COMMANDER DEATH
    // ========================================================

    if (
        reason ===
        "commander"
    ) {

        if (
            winner ===
            "red"
        ) {

            redReward = 1;
            blueReward = -1;

        } else {

            redReward = -1;
            blueReward = 1;
        }


        episodeReward =
            winner ===
            "red"
                ? redReward
                : blueReward;
    }


    // ========================================================
    // TIMEOUT
    // ========================================================

    else if (
        reason ===
        "timeout"
    ) {

        const redAlive =
            countAliveSoldiers(
                "red"
            );


        const blueAlive =
            countAliveSoldiers(
                "blue"
            );


        // 生存兵士数の差 / 100
        const rewardDifference =
            (
                redAlive -
                blueAlive
            ) / 100;


        redReward =
            rewardDifference;


        blueReward =
            -rewardDifference;


        // 勝敗
        if (
            redAlive >
            blueAlive
        ) {

            battleWinner =
                "red";

        } else if (
            blueAlive >
            redAlive
        ) {

            battleWinner =
                "blue";

        } else {

            battleWinner =
                "draw";
        }


        episodeReward =
            redReward;


        // 画面表示
        if (
            battleWinner ===
            "red"
        ) {

            document.getElementById(
                "status"
            ).innerHTML =
                "<br>⏱ TIMEOUT → 🏆 RED WINS!";

        } else if (
            battleWinner ===
            "blue"
        ) {

            document.getElementById(
                "status"
            ).innerHTML =
                "<br>⏱ TIMEOUT → 🏆 BLUE WINS!";

        } else {

            document.getElementById(
                "status"
            ).innerHTML =
                "<br>⏱ TIMEOUT → DRAW";
        }


        console.log(
            "===== TIMEOUT ====="
        );

        console.log(
            "Red soldiers:",
            redAlive
        );

        console.log(
            "Blue soldiers:",
            blueAlive
        );

        console.log(
            "Red reward:",
            redReward
        );

        console.log(
            "Blue reward:",
            blueReward
        );
    }


    // ========================================================
    // DEBUG
    // ========================================================

    if (
        reason ===
        "commander"
    ) {

        document.getElementById(
            "status"
        ).innerHTML =
            winner === "red"
                ? "<br>🏆 RED WINS!"
                : "<br>🏆 BLUE WINS!";
    }


    console.log(
        "===== EPISODE END ====="
    );

    console.log(
        "Reason:",
        reason
    );

    console.log(
        "Winner:",
        battleWinner
    );

    console.log(
        "Red reward:",
        redReward
    );

    console.log(
        "Blue reward:",
        blueReward
    );

    console.log(
        "Episode reward:",
        episodeReward
    );
}


// ============================================================
// EXECUTE COMMAND
// ============================================================

function executeCommand(
    unit,
    delta
) {

    if (
        !unit.userData.alive
    ) {
        return false;
    }


    // Attack freeze
    if (
        unit.userData.attackCooldown >
        0
    ) {
        return false;
    }


    const command =
        unit.userData.command;


    const speed =
        unit.userData.speed;


    const dx =
        command.dx *
        speed *
        delta;

    const dz =
        command.dz *
        speed *
        delta;


    const currentX =
        unit.position.x;

    const currentZ =
        unit.position.z;


    const radius =
        unit.userData.radius;


    const limit =
        FIELD_LIMIT -
        radius;


    // ========================================================
    // Direct movement
    // ========================================================

    const nx =
        currentX +
        dx;

    const nz =
        currentZ +
        dz;


    if (
        nx >= -limit &&
        nx <= limit &&
        nz >= -limit &&
        nz <= limit &&
        !collidesWithWall(
            nx,
            nz,
            radius
        ) &&
        !collidesWithUnit(
            nx,
            nz,
            radius,
            unit
        )
    ) {

        unit.position.x =
            nx;

        unit.position.z =
            nz;

        return true;
    }


    // ========================================================
    // X only
    // ========================================================

    const candidateX =
        currentX +
        dx;


    if (
        candidateX >= -limit &&
        candidateX <= limit &&
        !collidesWithWall(
            candidateX,
            currentZ,
            radius
        ) &&
        !collidesWithUnit(
            candidateX,
            currentZ,
            radius,
            unit
        )
    ) {

        unit.position.x =
            candidateX;

        return true;
    }


    // ========================================================
    // Z only
    // ========================================================

    const candidateZ =
        currentZ +
        dz;


    if (
        candidateZ >= -limit &&
        candidateZ <= limit &&
        !collidesWithWall(
            currentX,
            candidateZ,
            radius
        ) &&
        !collidesWithUnit(
            currentX,
            candidateZ,
            radius,
            unit
        )
    ) {

        unit.position.z =
            candidateZ;

        return true;
    }


    totalBlockedMoves++;

    return false;
}


// ============================================================
// ATTACK COMMAND
// ============================================================

function executeAttackCommand(
    unit
) {

    if (
        !unit.userData.command.attack
    ) {
        return;
    }


    if (
        unit.userData.attackCooldown >
        0
    ) {
        return;
    }


    const enemy =
        findNearestEnemy(
            unit
        );


    if (
        enemy.unit === null
    ) {
        return;
    }


    if (
        enemy.distance <=
        ATTACK_RANGE
    ) {

        attack(
            unit,
            enemy.unit
        );
    }
}


// ============================================================
// STUCK
// ============================================================

function updateStuck(
    unit,
    delta,
    moved
) {

    if (
        moved
    ) {

        unit.userData.stuckTime =
            0;

    } else {

        unit.userData.stuckTime +=
            delta;
    }
}


function countStuckUnits() {

    let count = 0;


    for (
        const unit
        of units
    ) {

        if (
            unit.userData.alive &&
            unit.userData.type ===
            "soldier" &&
            unit.userData.stuckTime >
            0.50
        ) {

            count++;
        }
    }


    return count;
}


// ============================================================
// DEBUG
// ============================================================

function updateDebug() {

    const redAlive =
        countAliveSoldiers(
            "red"
        );


    const blueAlive =
        countAliveSoldiers(
            "blue"
        );


    const elapsed =
        (
            performance.now() -
            battleStartTime
        ) / 1000;


    const remaining =
        Math.max(
            0,
            MAX_BATTLE_TIME -
            elapsed
        );


    document.getElementById(
        "redAlive"
    ).textContent =
        redAlive;


    document.getElementById(
        "blueAlive"
    ).textContent =
        blueAlive;


    document.getElementById(
        "redHP"
    ).textContent =
        Math.max(
            0,
            Math.round(
                redCommander.userData.hp
            )
        );


    document.getElementById(
        "blueHP"
    ).textContent =
        Math.max(
            0,
            Math.round(
                blueCommander.userData.hp
            )
        );


    document.getElementById(
        "attacks"
    ).textContent =
        totalAttacks;


    document.getElementById(
        "blocked"
    ).textContent =
        totalBlockedMoves;


    document.getElementById(
        "stuck"
    ).textContent =
        countStuckUnits();


    document.getElementById(
        "commandUpdates"
    ).textContent =
        commandUpdates;


    document.getElementById(
        "forwardPasses"
    ).textContent =
        forwardPasses;


    document.getElementById(
        "time"
    ).textContent =
        elapsed.toFixed(1);


    document.getElementById(
        "remaining"
    ).textContent =
        remaining.toFixed(1);


    document.getElementById(
        "redReward"
    ).textContent =
        redReward.toFixed(3);


    document.getElementById(
        "blueReward"
    ).textContent =
        blueReward.toFixed(3);


    document.getElementById(
        "episodeReward"
    ).textContent =
        episodeReward.toFixed(3);
}


// ============================================================
// FPS
// ============================================================

let fpsFrames = 0;

let fpsLastTime =
    performance.now();


function updateFPS() {

    fpsFrames++;


    const now =
        performance.now();


    if (
        now -
        fpsLastTime >=
        500
    ) {

        const fps =
            fpsFrames *
            1000 /
            (
                now -
                fpsLastTime
            );


        document.getElementById(
            "fps"
        ).textContent =
            fps.toFixed(1);


        fpsFrames = 0;

        fpsLastTime =
            now;
    }
}


// ============================================================
// ANIMATION
// ============================================================

let previousTime =
    performance.now();

let commandTimer =
    0;

let debugTimer =
    0;


function animate(time) {

    requestAnimationFrame(
        animate
    );


    const delta =
        Math.min(
            (
                time -
                previousTime
            ) / 1000,
            0.05
        );


    previousTime =
        time;


    if (
        !battleEnded
    ) {

        // ----------------------------------------------------
        // Time limit
        // ----------------------------------------------------

        const elapsed =
            (
                time -
                battleStartTime
            ) / 1000;


        if (
            elapsed >=
            MAX_BATTLE_TIME
        ) {

            endBattle(
                null,
                "timeout"
            );

        } else {

            // ------------------------------------------------
            // Update NN commands
            // ------------------------------------------------

            commandTimer -=
                delta;


            if (
                commandTimer <=
                0
            ) {

                updateNNCommands();

                commandTimer =
                    COMMAND_INTERVAL;
            }


            // ------------------------------------------------
            // Soldiers
            // ------------------------------------------------

            for (
                const unit
                of units
            ) {

                if (
                    !unit.userData.alive ||
                    unit.userData.type !==
                    "soldier"
                ) {
                    continue;
                }


                unit.userData.attackCooldown =
                    Math.max(
                        0,
                        unit.userData.attackCooldown -
                        delta
                    );


                const moved =
                    executeCommand(
                        unit,
                        delta
                    );


                updateStuck(
                    unit,
                    delta,
                    moved
                );


                executeAttackCommand(
                    unit
                );


                const command =
                    unit.userData.command;


                if (
                    Math.abs(
                        command.dx
                    ) +
                    Math.abs(
                        command.dz
                    ) >
                    0
                ) {

                    unit.rotation.y =
                        Math.atan2(
                            command.dx,
                            command.dz
                        );
                }
            }


            // ------------------------------------------------
            // Commanders
            // ------------------------------------------------

            for (
                const commander
                of [
                    redCommander,
                    blueCommander
                ]
            ) {

                if (
                    !commander.userData.alive
                ) {
                    continue;
                }


                commander.userData.attackCooldown =
                    Math.max(
                        0,
                        commander.userData.attackCooldown -
                        delta
                    );


                // 大将は現時点ではランダム移動
                if (
                    Math.random() <
                    0.02
                ) {

                    const angle =
                        Math.random() *
                        Math.PI *
                        2;


                    commander.userData.velocity.x =
                        Math.cos(angle);

                    commander.userData.velocity.z =
                        Math.sin(angle);
                }


                const v =
                    commander.userData.velocity;

                const speed =
                    commander.userData.speed;


                const nx =
                    commander.position.x +
                    v.x *
                    speed *
                    delta;

                const nz =
                    commander.position.z +
                    v.z *
                    speed *
                    delta;


                const r =
                    commander.userData.radius;

                const limit =
                    FIELD_LIMIT -
                    r;


                if (
                    nx >= -limit &&
                    nx <= limit &&
                    nz >= -limit &&
                    nz <= limit &&
                    !collidesWithWall(
                        nx,
                        nz,
                        r
                    ) &&
                    !collidesWithUnit(
                        nx,
                        nz,
                        r,
                        commander
                    )
                ) {

                    commander.position.x =
                        nx;

                    commander.position.z =
                        nz;
                }
            }
        }
    }


    // --------------------------------------------------------
    // DEBUG
    // --------------------------------------------------------

    debugTimer +=
        delta;


    if (
        debugTimer >=
        0.1
    ) {

        updateDebug();

        debugTimer =
            0;
    }


    updateFPS();


    renderer.render(
        scene,
        camera
    );
}


animate(
    performance.now()
);


// ============================================================
// RESIZE
// ============================================================

window.addEventListener(
    "resize",
    () => {

        camera.aspect =
            window.innerWidth /
            window.innerHeight;

        camera.updateProjectionMatrix();

        renderer.setSize(
            window.innerWidth,
            window.innerHeight
        );
    }
);

</script>
</body>
</html>'
    style="width:100%; height:700px; border:none;">
</iframe>
"""))